In [ ]:
import sys
from pathlib import Path

In [ ]:
def test_yolo_pt(image_path, model_path):
    print(f"\n--- Testing YOLO (.pt) ---")
    try:
        from ultralytics import YOLO
        model = YOLO(model_path)
        results = model(image_path)
        for r in results:
            print(f"Detected {len(r.boxes)} objects:")
            for box in r.boxes:
                class_id = int(box.cls[0])
                conf = float(box.conf[0])
                label = model.names[class_id]
                print(f" - {label}: {conf:.2f}")
    except Exception as e:
        print(f"Error testing .pt model: {e}")

In [ ]:
def test_yolo_tflite(image_path, model_path, labels_path):
    print(f"\n--- Testing YOLO (.tflite) ---")
    try:
        import numpy as np
        import tensorflow as tf
        import cv2
        import json
        
        # Load labels
        with open(labels_path, 'r', encoding='utf-8') as f:
            labels_data = json.load(f)
            if isinstance(labels_data, list):
                labels = labels_data
            elif isinstance(labels_data, dict) and 'labels' in labels_data:
                labels = labels_data['labels']
            else:
                labels = [f"class_{i}" for i in range(100)]
                
        # Load TFLite model
        interpreter = tf.lite.Interpreter(model_path=model_path)
        interpreter.allocate_tensors()
        
        input_details = interpreter.get_input_details()[0]
        output_details = interpreter.get_output_details()
        
        print(f"Input shape: {input_details['shape']}, dtype: {input_details['dtype']}")
        for out in output_details:
            print(f"Output name: {out['name']}, shape: {out['shape']}, dtype: {out['dtype']}")
            
        # Prepare image
        input_shape = input_details['shape'] # usually [1, 640, 640, 3] or [1, 3, 640, 640]
        h, w = input_shape[1:3] if input_shape[1] > 3 else input_shape[2:4]
        
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_resized = cv2.resize(img, (w, h))
        
        # YOLOv8 TFLite often expects float32 normalized 0-1, or uint8 depending on quantization
        if input_details['dtype'] == np.uint8:
            input_data = np.expand_dims(img_resized, axis=0).astype(np.uint8)
        else:
            input_data = (np.expand_dims(img_resized, axis=0) / 255.0).astype(np.float32)
            
        if input_shape[1] == 3: # NCHW
            input_data = np.transpose(input_data, (0, 3, 1, 2))
            
        interpreter.set_tensor(input_details['index'], input_data)
        interpreter.invoke()
        
        output_data = interpreter.get_tensor(output_details[0]['index'])
        print(f"Raw output shape: {output_data.shape}")
        
        # Just simple parse to show it ran
        # Usually YOLOv8 TFLite output is [1, num_classes + 4, num_boxes]
        if len(output_data.shape) == 3 and output_data.shape[1] > 4:
            boxes = output_data[0].transpose()
            valid_detections = 0
            for box in boxes:
                scores = box[4:]
                max_score = np.max(scores)
                if max_score > 0.25:
                    class_id = np.argmax(scores)
                    valid_detections += 1
            print(f"Found {valid_detections} boxes with conf > 0.25")
        
        print("TFLite inference ran successfully.")
    except Exception as e:
        print(f"Error testing .tflite model: {e}")

In [ ]:
image = "C:/Dev/projects/assist-eye/assets/model_testing/test_yolo.jpg"
pt = "C:/Dev/projects/assist-eye/assets/models/yolo/best.pt"
tflite = "C:/Dev/projects/assist-eye/assets/models/yolo/best_int8.tflite"
labels = "C:/Dev/projects/assist-eye/assets/models/yolo_labels.json"

In [ ]:
test_yolo_pt(image, pt)

In [ ]:
test_yolo_tflite(image, tflite, labels)